# Load the customers and orders 

In [0]:
from pyspark.sql.functions import *

customers_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customers/"

orders_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

df_customers = spark.read.format("delta").load(customers_silver_path)

df_orders = spark.read.format("delta").load(orders_silver_path)
display(df_customers)
display(df_orders)

# Aggregate orders by customer

In [0]:
customer_sales = (
    df_orders
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_spend"),
        avg("total_amount").alias("average_order_value"),
        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date")
    )
)
display(customer_sales)

# Add customer names

In [0]:
df_customer_sales = (
    customer_sales
    .join(
        df_customers.select(
            "customer_id",
            concat_ws(" ", col("first_name"), col("last_name")).alias("customer_name")
        ),
        on="customer_id",
        how="left"
    )
)

In [0]:
df_customer_sales = df_customer_sales.select(
    "customer_id",
    "customer_name",
    "total_orders",
    "total_quantity",
    "total_spend",
    "average_order_value",
    "first_order_date",
    "last_order_date"
)
display(df_customer_sales)

# Write to Gold

In [0]:
customer_sales_gold_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/gold/customer_sales/"

df_customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(customer_sales_gold_path)

# Verify the Gold table

In [0]:
df_customer_sales_gold = (
    spark.read
    .format("delta")
    .load(customer_sales_gold_path)
)

display(df_customer_sales_gold)
print("Customer Sales records:", df_customer_sales_gold.count())